# Lab 7: U.S. Patent Phrase to Phrase Matching
**Student:** Swarnim | **Roll No:** A24ARIU0004

Compare the course DeBERTa-v3-small regression baseline with an untuned embedding model and two context ablations. Keep the same held-out anchors throughout. This notebook adapts the [course starter](patent_phrase_starter.ipynb) for multiple controlled runs and an offline Kaggle submission.

## Setup and execution modes

For the full study, use a GPU and set `PATENT_MODE=study` (default outside Kaggle). Download the official competition files via Kaggle and set `PATENT_DATA_DIR` to their extracted directory. The first local run downloads two public pretrained checkpoints. Install `torch`, `transformers`, `accelerate`, `datasets`, `sentencepiece`, `pandas`, `numpy`, and `scikit-learn` if missing.

On Kaggle, attach the competition as an input and attach the original DeBERTa checkpoint in a `deberta-v3-small` directory. This run uses [mtnash/deberta-v3-small, Transformers/default v1](https://www.kaggle.com/models/mtnash/deberta-v3-small/transformers/default/1). Its weights and SentencePiece file match the original Hugging Face download by SHA-256. The course [download helper](download_weights.ipynb) is another way to attach the same model. Select one GPU and turn internet off. The default Kaggle mode is `submit`, which retrains only the chosen experiment on the original training split and writes `/kaggle/working/submission.csv`. To run the full study on Kaggle, select `study` and also attach the MiniLM checkpoint. The two CSVs produced by the study are local run artifacts.

In [1]:
import gc
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
if Path("/kaggle/input").is_dir():
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["HF_HUB_OFFLINE"] = "1"

import torch
import torch.nn.functional as F
import transformers
from datasets import Dataset, disable_progress_bar
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModel, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED, VAL_FRACTION = 42, 0.2
MAX_LEN, EPOCHS, LR, BATCH_SIZE = 64, 3, 2e-5, 32
MODEL_ID = "microsoft/deberta-v3-small"
EMBED_ID = "sentence-transformers/all-MiniLM-L6-v2"
ON_KAGGLE = Path("/kaggle/input").is_dir()
MODE = os.getenv("PATENT_MODE", "submit" if ON_KAGGLE else "study")
FINAL_VARIANT = "raw"  # selected by local validation; see the results below
assert MODE in {"study", "submit"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE = Path.home() / ".cache" / "swarnim_labs" / "patent_phrase_matching"
OUT = Path("/kaggle/working") if ON_KAGGLE else CACHE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
disable_progress_bar()
transformers.set_seed(SEED)
print(f"Mode: {MODE} | device: {DEVICE} | torch: {torch.__version__} | transformers: {transformers.__version__}")

def model_path(repo_id):
    if not ON_KAGGLE:
        return repo_id
    model_name = repo_id.rsplit("/", 1)[-1].lower()
    for config in sorted(Path("/kaggle/input").rglob("config.json")):
        folder = config.parent
        names = [part.lower() for part in folder.parts]
        if model_name in names and any(folder.glob("*.safetensors")):
            return str(folder)
        if model_name in names and (folder / "pytorch_model.bin").is_file():
            return str(folder)
    raise FileNotFoundError(f"Attach a local {repo_id} checkpoint with tokenizer and weights.")

Mode: study | device: cuda | torch: 2.7.0+cu126 | transformers: 5.7.0


In [2]:
def data_path():
    if "PATENT_DATA_DIR" in os.environ:
        return Path(os.environ["PATENT_DATA_DIR"])
    if ON_KAGGLE:
        for path in sorted(Path("/kaggle/input").rglob("train.csv")):
            if (path.parent / "test.csv").is_file() and (path.parent / "sample_submission.csv").is_file():
                fields = set(pd.read_csv(path, nrows=0).columns)
                if {"id", "anchor", "target", "context", "score"} <= fields:
                    return path.parent
    return CACHE / "data"

data = data_path()
train = pd.read_csv(data / "train.csv", dtype={"id": str})
test = pd.read_csv(data / "test.csv", dtype={"id": str})
sample = pd.read_csv(data / "sample_submission.csv", dtype={"id": str})
assert train.id.is_unique and test.id.is_unique
assert set(train.score.unique()) <= {0, 0.25, 0.5, 0.75, 1}
if not ON_KAGGLE:
    assert set(sample.id) == set(test.id)

splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn = train.iloc[train_idx].reset_index(drop=True)
val = train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"Training rows: {len(trn):,} | validation rows: {len(val):,} | validation anchors: {val.anchor.nunique()}")
print(train.score.value_counts().sort_index().rename("Training label counts").to_string())

Training rows: 28,914 | validation rows: 7,559 | validation anchors: 147
score
0.00     7471
0.25    11519
0.50    12300
0.75     4029
1.00     1154


## Task 1 and Task 3 — One split, three cross-encoder runs

A random row split can give both partitions pairs with the same anchor. The model can learn anchor-specific patterns during training, so a random split can overestimate performance on new anchors. Grouping on `anchor` holds out every row for a selected anchor and supports a fair comparison across runs.

The baseline follows the starter's architecture, pair template, split, and training hyperparameters. The two ablations change only the context suffix: use the raw CPC code or remove context. Each run starts at the same pretrained weights and seed. Validation Pearson uses raw predictions, as in the starter. Timing starts at training and includes epoch validation, but excludes download, tokenization and the final prediction pass.

In [3]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}
VARIANTS = {
    "section": "CPC section name (starter)",
    "raw": "Raw CPC code",
    "none": "No context",
}

def pearson(actual, predicted):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    if len(actual) < 2 or np.ptp(actual) == 0 or np.ptp(predicted) == 0:
        return np.nan
    return float(np.corrcoef(actual, predicted)[0, 1])

def compute_metrics(eval_pred):
    return {"pearson": pearson(eval_pred.label_ids, eval_pred.predictions.reshape(-1))}

def pair_tokenizer(checkpoint):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=ON_KAGGLE)
    tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
        single="[CLS] $A [SEP]",
        pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)],
    )
    return tokenizer

def tokenize_pairs(frame, tokenizer, variant):
    targets = frame.target
    if variant == "section":
        context = frame.context.str[0].map(CPC_SECTIONS).fillna(frame.context)
        targets = targets + "; " + context
    elif variant == "raw":
        targets = targets + "; " + frame.context
    elif variant != "none":
        raise ValueError(f"Unknown context variant: {variant}")
    tokens = tokenizer(frame.anchor.tolist(), targets.tolist(), max_length=MAX_LEN, truncation=True)
    dataset = dict(tokens)
    if "score" in frame:
        dataset["labels"] = frame.score.astype("float32").tolist()
    return Dataset.from_dict(dataset)

def fit_variant(variant):
    transformers.set_seed(SEED)
    ckpt = model_path(MODEL_ID)
    tokenizer = pair_tokenizer(ckpt)
    ds_train = tokenize_pairs(trn, tokenizer, variant)
    ds_val = tokenize_pairs(val, tokenizer, variant)
    model = AutoModelForSequenceClassification.from_pretrained(
        ckpt, num_labels=1, dtype=torch.float32, local_files_only=ON_KAGGLE,
    )
    args = TrainingArguments(
        output_dir=str(OUT / f"trainer_{variant}"),
        num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=2 * BATCH_SIZE,
        warmup_steps=math.ceil(0.1 * EPOCHS * math.ceil(len(ds_train) / BATCH_SIZE)), weight_decay=0.01, eval_strategy="epoch",
        save_strategy="no", logging_steps=100, fp16=DEVICE == "cuda",
        report_to="none", seed=SEED, disable_tqdm=True,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
        data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,
    )
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    trainer.train()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    peak = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else None
    val_pred = trainer.predict(ds_val).predictions.reshape(-1)
    test_pred = trainer.predict(tokenize_pairs(test, tokenizer, variant)).predictions.reshape(-1)
    metrics = dict(variant=variant, setting=VARIANTS[variant], pearson=pearson(val.score, val_pred),
                   seconds=seconds, peak_gb=peak)
    del trainer, model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return metrics, val_pred, test_pred

## Task 2 — Frozen bi-encoder

MiniLM encodes unique phrases independently. Mean-pool the contextual token embeddings with the attention mask and normalize each vector. Their dot product is cosine similarity. The cross-encoder instead uses joint attention over the two phrases and learns a regression head from labels. The observed gap combines supervision, joint encoding, context (in the starter), and checkpoint differences.

In [4]:
@torch.inference_mode()
def frozen_predictions(frame):
    ckpt = model_path(EMBED_ID)
    tokenizer = AutoTokenizer.from_pretrained(ckpt, local_files_only=ON_KAGGLE)
    encoder = AutoModel.from_pretrained(ckpt, local_files_only=ON_KAGGLE).eval().to(DEVICE)
    phrases = pd.Index(pd.unique(pd.concat([frame.anchor, frame.target], ignore_index=True)))
    pieces = []
    for start in range(0, len(phrases), 128):
        batch = tokenizer(phrases[start:start+128].tolist(), return_tensors="pt",
                          padding=True, truncation=True, max_length=256).to(DEVICE)
        tokens = encoder(**batch).last_hidden_state
        mask = batch.attention_mask.unsqueeze(-1)
        mean = (tokens * mask).sum(1) / mask.sum(1).clamp_min(1)
        pieces.append(F.normalize(mean, dim=1).cpu())
    vectors = torch.cat(pieces)
    a, b = phrases.get_indexer(frame.anchor), phrases.get_indexer(frame.target)
    return (vectors[a] * vectors[b]).sum(1).numpy()

In [5]:
runs = {}
if MODE == "study":
    start = time.perf_counter()
    bi = frozen_predictions(val)
    bi_metrics = dict(variant="bi", setting="Frozen MiniLM cosine", pearson=pearson(val.score, bi),
                      seconds=0.0, evaluation_seconds=time.perf_counter()-start, peak_gb=None)
    print(f"Frozen MiniLM validation Pearson: {bi_metrics['pearson']:.4f}", flush=True)
    for name in VARIANTS:
        print(f"\nTraining {VARIANTS[name]}", flush=True)
        runs[name] = fit_variant(name)
        print(json.dumps(runs[name][0]), flush=True)
    best = max(runs, key=lambda name: runs[name][0]["pearson"])
    result_rows = [bi_metrics] + [run[0] for run in runs.values()]
    (OUT / "results.json").write_text(json.dumps({"best": best, "runs": result_rows}, indent=2))
    display(pd.DataFrame(result_rows).drop(columns="evaluation_seconds").round(4))
else:
    best = FINAL_VARIANT
    runs[best] = fit_variant(best)

metrics, predictions, test_predictions = runs[best]
print(f"Selected {metrics['setting']} | validation Pearson {metrics['pearson']:.4f}")

Frozen MiniLM validation Pearson: 0.5814



Training CPC section name (starter)


[transformers] The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.de

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


{'loss': '0.1251', 'grad_norm': '1.269', 'learning_rate': '7.206e-06', 'epoch': '0.1106'}


... (98 outputs left out)

{'eval_loss': '0.02434', 'eval_pearson': '0.8169', 'eval_runtime': '1.587', 'eval_samples_per_second': '4764', 'eval_steps_per_second': '75', 'epoch': '3'}
{'train_runtime': '270.9', 'train_samples_per_second': '320.3', 'train_steps_per_second': '10.01', 'train_loss': '0.02977', 'epoch': '3'}


{"variant": "none", "setting": "No context", "pearson": 0.816860716385301, "seconds": 270.99629046500195, "peak_gb": 2.876202496}


,variant,setting,pearson,seconds,peak_gb
0,bi,Frozen MiniLM cosine,0.5814,0.0000,NaN
1,section,CPC section name (starter),0.8178,312.5556,2.8737
2,raw,Raw CPC code,0.8195,286.7392,2.8761
3,none,No context,0.8169,270.9963,2.8762


Selected Raw CPC code | validation Pearson 0.8195


### Measured validation results

| Setting | Validation Pearson | Training time | Interpretation |
| --- | ---: | ---: | --- |
| Frozen MiniLM cosine | 0.5814 | 0.0 s | Lowest correlation on the same held-out anchors. |
| CPC section name (starter) | 0.8178 | 312.6 s | Reference configuration from the starter. |
| Raw CPC code | 0.8195 | 286.7 s | Highest observed score, by 0.0017 over the starter. |
| No context | 0.8169 | 271.0 s | Only 0.0009 below the starter and fastest of the supervised settings. |

The cross-encoder with the starter context exceeded frozen MiniLM by 0.2364 Pearson. The experiment compares a different encoder, a joint input, and task supervision, so the gap cannot isolate the effect of fine-tuning alone. The measured runs used one RTX 2000 Ada GPU, Transformers 5.7.0, and the course starter's 10% warmup. The Kaggle version expresses the same warmup as an explicit step count for compatibility.

Raw CPC was selected because it had the highest validation Pearson (0.8195). The three supervised scores span only 0.0026, below the brief's 0.005 noise guideline. This single-seed result supports a selection for submission, but cannot establish a reliable benefit from changing the context suffix.

## Task 4 — Validation errors

Group the selected model's predictions by CPC section and true score. Pearson is undefined if a group has fewer than two rows or zero variance. The per-score table uses raw predictions to reveal calibration and regression toward the mean. The individual cases are saved locally for inspection.

In [6]:
validation = val[["id", "anchor", "target", "context", "score"]].assign(pred=predictions)
validation["section"] = validation.context.str[0]
validation["absolute_error"] = (validation.score - validation.pred).abs()
sections = pd.DataFrame([
    {"section": section, "rows": len(group), "pearson": pearson(group.score, group.pred)}
    for section, group in validation.groupby("section")
]).set_index("section").reindex(list(CPC_SECTIONS))
score_summary = validation.groupby("score").pred.agg(count="size", mean_prediction="mean")
worst = validation.nlargest(10, "absolute_error")
if MODE == "study":
    validation.to_csv(OUT / "val_predictions.csv", index=False)
    worst.to_csv(OUT / "largest_errors.csv", index=False)
    print("Saved the ten largest errors for local inspection (largest_errors.csv).")
display(sections.round(4), score_summary.round(4))

,rows,pearson
section,,
A,943,0.7778
B,1926,0.8984
C,1177,0.7663
D,199,0.7599
E,373,0.8115
F,968,0.8489
G,1040,0.6996
H,933,0.8091


,count,mean_prediction
score,,
0.00,1522,0.1104
0.25,2465,0.2988
0.50,2475,0.4815
0.75,832,0.6526
1.00,265,1.0108


Saved the ten largest errors for local inspection (largest_errors.csv).


### Interpretation

G (physics) was lowest at 0.6996 across 1,040 rows, followed by D (textiles and paper) at 0.7599 across 199. Several of the largest G errors mix superficially related optics or computing terms. Less common anchors and ambiguous domain wording are plausible contributors. D's small holdout makes that estimate more sensitive to a few anchors. B was highest at 0.8984 across 1,926 rows.

The model pushes label 0 up to a mean of 0.1104 and pulls 0.75 down to 0.6526. The 0.25 and 0.50 categories supply most validation rows. The 1.0 group is rare (265 rows), yet its mean is 1.0108, so compression is not uniform. The unconstrained regression head can overshoot. Submission scores are clipped to [0, 1].

Eight of the ten largest errors are overpredictions on score-0 rows. Six involve repeated or near-identical wording that changes role, order, or structure, and two involve related technical concepts the model treats as interchangeable. The two score-0.75 misses require linking specific domain terms with little obvious overlap. Some reordered or slightly rephrased cases could appear equivalent without the underlying patent passage, so those zero labels merit review as ambiguous examples. The supplied labels remain unchanged.

The exact ten rows are available in the local diagnostic CSV and the local report. The repository notebook records aggregate findings.

## Task 5 — Competition output

The submission keeps test order, checks the ID set, and clips scores to the competition's label range. The visible test rows are a file-format check. Kaggle substitutes its hidden test set when scoring an offline notebook version.

In [7]:
submission = pd.DataFrame({"id": test.id, "score": np.clip(test_predictions, 0, 1)})
assert list(submission.columns) == ["id", "score"]
assert submission.id.tolist() == test.id.tolist() and submission.id.is_unique
assert np.isfinite(submission.score).all() and submission.score.between(0, 1).all()
submission.to_csv(OUT / "submission.csv", index=False)
print(f"Wrote {len(submission):,} predictions with columns id, score.")

Wrote 36 predictions with columns id, score.


### Submission and score comparison

Kaggle Version 1 was queued with internet disabled. Public and private leaderboard scores are pending its run and late submission.

[Kaggle notebook version](https://www.kaggle.com/code/contentiousplatypus/lab-7-patent-phrase-matching-a24ariu0004).

## References

- [Course project brief](project_brief.md) and [starter notebook](patent_phrase_starter.ipynb), adapted for the grouped evaluation and context ablations.
- [Kaggle competition data and scoring](https://www.kaggle.com/competitions/us-patent-phrase-to-phrase-matching)
- [DeBERTa-v3-small checkpoint](https://huggingface.co/microsoft/deberta-v3-small) and [all-MiniLM-L6-v2 checkpoint and pooling recipe](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).